In [ ]:
from jaxtyping import Float, Array, Key, Scalar
import jax
import jax.numpy as jnp
import jax.scipy as jsp
import jax.random as jr
from jax.numpy.fft import rfftfreq, irfft, rfft, fft
from flax import nnx
import optax
from einops import rearrange, einsum

import numpy as np
from tqdm import tqdm
from matplotlib import pyplot as plt
from corner import corner
from emcee import EnsembleSampler

from jax import config
config.update("jax_enable_x64",True)
rngs = nnx.Rngs(42)

In [ ]:
# problem definition
SUN_EARTH_RADIUS = 1.496e11
SPEED_OF_LIGHT = 3e8

ARM_LENGTH = 2.5e9
ALPHA_TRIANGLE = 0   # Angle between earth-sun and lisa-sun directions, see Appendix A of https://arxiv.org/abs/2312.10792
BETA_TRIANGLE = 0    # Initial orientation of lisa constellation, see Appendix A of https://arxiv.org/abs/2312.10792

YEARSECONDS = 365 * 24 * 3600
WEEKSECONDS = 7 * 24 * 3600
YEAROMEGA = 2 * np.pi / YEARSECONDS

NTIMESAMPS = WEEKSECONDS // 50
TOBS = WEEKSECONDS
DT = TOBS / NTIMESAMPS
TIMES = jnp.linspace(0, TOBS, NTIMESAMPS, endpoint=False)


Parameters = Float[Array, "sources 6"]
Observation = Float[Array, "times 2"]

PARAMETERS = 6
CHANNELS = 2
SOURCES = 2
AMPLITUDE_RANGE = (1e-25, 1e-24)
COS_IOTA_RANGE = (-1, 1)
F0_RANGE = (1e-4, 1e-2)
PHI0_RANGE = (0, 2 * np.pi)
THETA_RANGE = (0, np.pi)
PHI_RANGE = (0, 2 * np.pi)

# architecture and training hyperparameters
NUM_BLOCKS = 1 # 4
NUM_HEADS = 2 # 8
HIDDEN_DIM = NUM_HEADS * 32
PATCH_SIZE = 512
LEARNING_RATE = 3e-4
BATCH_SIZE = 256
TOTAL_EXAMPLES = 1000 # 1024 * 1000

In [ ]:
POSITION_SAT1 = jnp.stack(
    [
        jnp.cos(YEAROMEGA * TIMES + ALPHA_TRIANGLE) + ARM_LENGTH / SUN_EARTH_RADIUS / (4 * jnp.sqrt(3)) * (jnp.cos(2 * YEAROMEGA * TIMES + ALPHA_TRIANGLE + BETA_TRIANGLE) - 3 * jnp.cos(ALPHA_TRIANGLE - BETA_TRIANGLE)),
        jnp.sin(YEAROMEGA * TIMES + ALPHA_TRIANGLE) + ARM_LENGTH / SUN_EARTH_RADIUS / (4 * jnp.sqrt(3)) * (jnp.sin(2 * YEAROMEGA * TIMES + ALPHA_TRIANGLE + BETA_TRIANGLE) - 3 * jnp.sin(ALPHA_TRIANGLE - BETA_TRIANGLE)),
        -ARM_LENGTH / SUN_EARTH_RADIUS / 2 * jnp.cos(YEAROMEGA * TIMES + BETA_TRIANGLE),
    ]
)

POSITION_SAT2 = jnp.stack(
    [
        jnp.cos(YEAROMEGA * TIMES + ALPHA_TRIANGLE) + ARM_LENGTH / SUN_EARTH_RADIUS / (4 * jnp.sqrt(3)) * (jnp.cos(2 * YEAROMEGA * TIMES + ALPHA_TRIANGLE + BETA_TRIANGLE - 2 * jnp.pi / 3) - 3 * jnp.cos(ALPHA_TRIANGLE - BETA_TRIANGLE + 2 * jnp.pi / 3)),
        jnp.sin(YEAROMEGA * TIMES + ALPHA_TRIANGLE) + ARM_LENGTH / SUN_EARTH_RADIUS / (4 * jnp.sqrt(3)) * (jnp.sin(2 * YEAROMEGA * TIMES + ALPHA_TRIANGLE + BETA_TRIANGLE - 2 * jnp.pi / 3) - 3 * jnp.sin(ALPHA_TRIANGLE - BETA_TRIANGLE + 2 * jnp.pi / 3)),
        -ARM_LENGTH / SUN_EARTH_RADIUS / 2 * jnp.cos(YEAROMEGA * TIMES + BETA_TRIANGLE - 2 * jnp.pi / 3),
    ]
)
POSITION_SAT3 = jnp.stack(
    [
        jnp.cos(YEAROMEGA * TIMES + ALPHA_TRIANGLE) + ARM_LENGTH / SUN_EARTH_RADIUS / (4 * jnp.sqrt(3)) * (jnp.cos(2 * YEAROMEGA * TIMES + ALPHA_TRIANGLE + BETA_TRIANGLE + 2 * jnp.pi / 3) - 3 * jnp.cos(ALPHA_TRIANGLE - BETA_TRIANGLE - 2 * jnp.pi / 3)),
        jnp.sin(YEAROMEGA * TIMES + ALPHA_TRIANGLE) + ARM_LENGTH / SUN_EARTH_RADIUS / (4 * jnp.sqrt(3)) * (jnp.sin(2 * YEAROMEGA * TIMES + ALPHA_TRIANGLE + BETA_TRIANGLE + 2 * jnp.pi / 3) - 3 * jnp.sin(ALPHA_TRIANGLE - BETA_TRIANGLE - 2 * jnp.pi / 3)),
        -ARM_LENGTH / SUN_EARTH_RADIUS / 2 * jnp.cos(YEAROMEGA * TIMES + BETA_TRIANGLE + 2 * jnp.pi / 3),
    ]
)

# tensor of arm directions: s(t) = d^ij(t) h_ij(t)
def tensor_product(a, b=None):
    return einsum(a, b if b is not None else a, "i k, j k -> i j k")


D_ALL = jnp.stack([
    tensor_product(POSITION_SAT1-POSITION_SAT2).T - tensor_product(POSITION_SAT1-POSITION_SAT3).T,
    tensor_product(POSITION_SAT2-POSITION_SAT3).T - tensor_product(POSITION_SAT2-POSITION_SAT1).T,
    tensor_product(POSITION_SAT3-POSITION_SAT1).T - tensor_product(POSITION_SAT3-POSITION_SAT2).T,
])/ (2 * (ARM_LENGTH / SUN_EARTH_RADIUS)**2)

# basis vectors that diagonalize the noise
D_A = (2 / jnp.sqrt(3)) * D_ALL[0] 
D_E = -(2 / 3) * D_ALL[0] - (4 / 3) * D_ALL[1]


def noise_psd(
        f: Float[Array, "..."], 
        acceleration_noise: float = 3.0, 
        interferometry_noise: float = 15.0, 
        channel: str = "E"
    ) -> Float[Array, "..."]:
    fstar = 1 / (2 * jnp.pi * ARM_LENGTH / SPEED_OF_LIGHT)
    if channel == "E":
        psd = (
            1 / 2 * (2 + jnp.cos(f / fstar)) * (interferometry_noise / ARM_LENGTH) ** 2 * 10 ** (-24) * (1 + (0.002 / f) ** 4)
            + 2 * (1 + jnp.cos(f / fstar) + (jnp.cos(f / fstar)) ** 2) * (acceleration_noise / ARM_LENGTH) ** 2 * 10 ** (-30)
            * (1 + (0.0004 / f) ** 2) * (1 + (f / 0.008) ** 4) * (1 / (2 * jnp.pi * f)) ** 4
        )
    elif channel == "T":
        psd = (
            (1 - jnp.cos(f / fstar)) * (interferometry_noise / ARM_LENGTH) ** 2 * 10 ** (-24) * (1 + (0.002 / f) ** 4) 
            + 2 * (1 - jnp.cos(f / fstar)) ** 2 * (acceleration_noise / ARM_LENGTH) ** 2 * 10 ** (-30) * (1 + (0.0004 / f) ** 2) 
            * (1 + (f / 0.008) ** 4) * (1 / (2 * jnp.pi * f)) ** 4
        )
    else:
        raise ValueError("Channel must be 'E' or 'T'")
    return jnp.where(f != 0, psd, 0.0)


def sample_noise(rng: Key) -> Float[Array, "time channel"]:
    psd = noise_psd(rfftfreq(NTIMESAMPS, DT), channel="E")
    real, imag = jr.normal(rng, shape=(2, NTIMESAMPS // 2 + 1, CHANNELS))
    xf = jnp.sqrt(psd[:, None] * NTIMESAMPS / 2.0) * (real + 1j * imag)
    return irfft(xf, axis=0)


### Polarization tensors ###
def phat(theta, phi):
    return jnp.stack([jnp.sin(phi), -jnp.cos(phi), 0 * phi])

def qhat(theta, phi):
    return jnp.stack(
        [jnp.cos(theta) * jnp.cos(phi), jnp.cos(theta) * jnp.sin(phi), -jnp.sin(theta)]
    )

def e_plus(theta, phi):
    # This agrees with 2201.08782 and 2009.11845 and is different wrt Allen-Ottewill
    return (tensor_product(phat(theta, phi), phat(theta, phi)) - tensor_product(qhat(theta, phi), qhat(theta, phi)))/ jnp.sqrt(2)  

def e_cross(theta, phi):
    # This agrees with 2201.08782 and 2009.11845 and is different wrt Allen-Ottewill
    return (tensor_product(phat(theta, phi), qhat(theta, phi)) + tensor_product(qhat(theta, phi), phat(theta, phi))) / jnp.sqrt(2) 


def sample_joint(rng: Key) -> tuple[Parameters, Observation]:
    rng_A, rng_cos_iota, rng_f0, rng_phi0, rng_theta, rng_phi, rng_noise = jr.split(rng, 7)
    log_A = jr.uniform(
        rng_A,
        shape=(SOURCES,),
        minval=np.log(AMPLITUDE_RANGE[0]),
        maxval=np.log(AMPLITUDE_RANGE[1]),
    )
    cos_iota = jr.uniform(
        rng_cos_iota,
        shape=(SOURCES,),
        minval=COS_IOTA_RANGE[0],
        maxval=COS_IOTA_RANGE[1],
    )
    log_f0 = jr.uniform(
        rng_f0,
        shape=(SOURCES,),
        minval=np.log(F0_RANGE[0]),
        maxval=np.log(F0_RANGE[1]),
    )
    phi0 = jr.uniform(
        rng_phi0,
        shape=(SOURCES,),
        minval=PHI0_RANGE[0],
        maxval=PHI0_RANGE[1],
    )

    theta = jr.uniform(
        rng_theta,
        shape=(SOURCES,),
        minval=THETA_RANGE[0],
        maxval=THETA_RANGE[1],
    )

    phi = jr.uniform(
        rng_phi,
        shape=(SOURCES,),
        minval=PHI_RANGE[0],
        maxval=PHI_RANGE[1],
    )

    A = jnp.exp(log_A)
    f0 = jnp.exp(log_f0)
    x = jnp.stack([log_A, cos_iota, log_f0, phi0, theta, phi], axis=-1)

    Phi_GB = 2 * jnp.pi * f0 * TIMES[..., None] - phi0
    h_plus = A[None] * (1 + cos_iota[None] ** 2) * jnp.cos(Phi_GB)
    h_cross = 2 * A[None] * cos_iota[None] * jnp.sin(Phi_GB)

    F_plus_A = jnp.einsum("tij, ijs -> ts", D_A, e_plus(theta, phi))
    F_cross_A = jnp.einsum("tij, ijs -> ts", D_A, e_cross(theta, phi))

    F_plus_E = jnp.einsum("tij, ijs -> ts", D_E, e_plus(theta, phi))
    F_cross_E = jnp.einsum("tij, ijs -> ts", D_E, e_cross(theta, phi))

    h_A = (h_plus * F_plus_A + h_cross * F_cross_A).sum(-1)
    h_E = (h_plus * F_plus_E + h_cross * F_cross_E).sum(-1)

    h = jnp.stack([h_A, h_E], axis=-1)
    #y = h + sample_noise(rng_noise)
    y = h + jr.normal(rng_noise, shape=h.shape)*1e-24
    return x, y

@jax.jit
def log_posterior(x_flat: Parameters, y: Observation) -> Scalar:
    x = rearrange(x_flat, "... (sources p) -> ... sources p", p=6)
    log_A, cos_iota, log_f0, phi0, theta, phi = x[..., 0], x[..., 1], x[..., 2], x[..., 3], x[..., 4], x[..., 5]
    A, f0 = jnp.exp(log_A), jnp.exp(log_f0)

    Phi_GB = 2 * jnp.pi * f0 * TIMES[..., None] - phi0
    h_plus = A[None] * (1 + cos_iota[None] ** 2) * jnp.cos(Phi_GB)
    h_cross = 2 * A[None] * cos_iota[None] * jnp.sin(Phi_GB)

    F_plus_A = jnp.einsum("tij, ijs -> ts", D_A, e_plus(theta, phi))
    F_cross_A = jnp.einsum("tij, ijs -> ts", D_A, e_cross(theta, phi))

    F_plus_E = jnp.einsum("tij, ijs -> ts", D_E, e_plus(theta, phi))
    F_cross_E = jnp.einsum("tij, ijs -> ts", D_E, e_cross(theta, phi))

    h_A = (h_plus * F_plus_A + h_cross * F_cross_A).sum(-1)
    h_E = (h_plus * F_plus_E + h_cross * F_cross_E).sum(-1)

    h = jnp.stack([h_A, h_E], axis=-1)

    #noisevars= noise_psd_on_channel_E(jnp.abs(fftfreq(NTIMESAMPS,TOBS/NTIMESAMPS)))[1:,None]*NTIMESAMPS
    noisevars = NTIMESAMPS * 1e-24**2
    residual = jnp.abs(fft(y - h,axis=0)[1:])
    log_likelihood = -einsum(residual**2 / (2*noisevars), "... t c-> ...") 

    mask_a = (AMPLITUDE_RANGE[0] < A) * (A < AMPLITUDE_RANGE[1])
    mask_f0 = (F0_RANGE[0] < f0) * (f0 < F0_RANGE[1])
    mask_phi = (PHI_RANGE[0] < phi) * (phi < PHI_RANGE[1])
    mask_cos_iota = (COS_IOTA_RANGE[0] < cos_iota) * (cos_iota < COS_IOTA_RANGE[1])
    mask_theta = (THETA_RANGE[0] < theta) * (theta < THETA_RANGE[1])
    mask_phi0 = (PHI0_RANGE[0] < phi0) * (phi0 < PHI0_RANGE[1])
    log_prior = jnp.where(
        mask_a * mask_f0 * mask_phi * mask_cos_iota * mask_theta * mask_phi0,  -log_A - log_f0, -jnp.inf
    ).sum(-1)
    return log_prior + log_likelihood

# TEST with white noise

In [ ]:
# psd sanity check
f=rfftfreq(NTIMESAMPS, DT)
plt.figure(figsize=(8,4))
plt.loglog(f, jnp.abs(rfft(sample_noise(rngs()), axis=0)))
plt.loglog(f, jnp.sqrt(NTIMESAMPS*noise_psd(f, channel="E")),"k")
plt.grid()
plt.show()

In [ ]:
# preprocessing sanity check
PATCH_SIZE = 512
x, y = sample_joint(rngs())
_, _, c = jsp.signal.stft(y, nperseg=PATCH_SIZE - 1, axis=-2, boundary=None)
c = rearrange(c, "... F C T -> C ... T F")
c1 = jnp.log(jnp.abs(c))
c2 = jnp.angle(c)

plt.figure(figsize=(10, 8))

plt.subplot(4, 1, 1)
plt.title("Log-amplitude spectrogram channel 1")
plt.imshow(c1[0].T, aspect="auto", origin="lower")
plt.colorbar()

plt.subplot(4, 1, 3)
plt.title("Angle spectrogram channel 1")
plt.imshow(c2[0].T, aspect="auto", origin="lower", cmap="hsv")
plt.colorbar()

plt.subplot(4, 1, 2)
plt.title("Log-amplitude spectrogram channel 2")
plt.imshow(c1[1].T, aspect="auto", origin="lower")
plt.colorbar()

plt.subplot(4, 1, 4)
plt.title("Angle spectrogram channel 2")
plt.imshow(c2[1].T, aspect="auto", origin="lower", cmap="hsv")
plt.colorbar()

plt.tight_layout()
plt.show()

In [ ]:
# architecture definition

def adaptive_norm(
    x: Float[Array, "... N D"],
    scale: Float[Array, "... 1 D"],
    shift: Float[Array, "... 1 D"],
):
    x = x - x.mean(axis=-1, keepdims=True)
    x = x / x.std(axis=-1, keepdims=True)
    x = x * (1 + scale) + shift
    return x


class Modulation(nnx.Module):
    def __init__(self, dim: int, *, rngs: nnx.Rngs):
        self.linear = nnx.LinearGeneral(
            in_features=dim,
            out_features=(1, 3 * dim),
            kernel_init=nnx.initializers.zeros,
            bias_init=nnx.initializers.zeros,
            rngs=rngs,
        )

    def __call__(self, y: Float[Array, "... D"]) -> tuple[Float[Array, "... 1 D"], ...]:
        modulation = self.linear(nnx.silu(y))
        shift, scale, gate = jnp.split(modulation, 3, axis=-1)
        return shift, scale, gate


class SinusoidalEmbed(nnx.Module):
    def __init__(self, dim: int, period: float = 2 * np.pi, *, rngs: nnx.Rngs):
        self.dim = dim
        self.period = period
        self.embed = FeedForward(2 * dim, dim, dim, rngs=rngs)

    def __call__(self, t: Float[Array, "..."]) -> Float[Array, "... D"]:
        freqs = jnp.exp(-jnp.log(self.period) * jnp.linspace(0, 1, self.dim))
        angles = 2 * jnp.pi * freqs * t[..., None]
        x = jnp.concat([jnp.sin(angles), jnp.cos(angles)], axis=-1)
        x = self.embed(x)
        return x


class FeedForward(nnx.Sequential):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        output_dim: int,
        activation=nnx.silu,
        *,
        rngs: nnx.Rngs,
    ):
        super().__init__(
            nnx.Linear(input_dim, hidden_dim, rngs=rngs),
            activation,
            nnx.Linear(hidden_dim, output_dim, rngs=rngs),
        )


class CrossAttention(nnx.Module):
    def __init__(self, dim: int, num_heads: int, use_bias=False, *, rngs: nnx.Rngs):
        super().__init__()
        assert dim % num_heads == 0, "dim should be divisible by num_heads"
        self.num_heads = num_heads
        self.qkv_proj_x = nnx.Linear(dim, dim * 3, use_bias=use_bias, rngs=rngs)
        self.qkv_proj_c = nnx.Linear(dim, dim * 3, use_bias=use_bias, rngs=rngs)
        self.out_proj_x = nnx.Linear(dim, dim, use_bias=use_bias, rngs=rngs)
        self.out_proj_c = nnx.Linear(dim, dim, use_bias=use_bias, rngs=rngs)

    def __call__(self, x: Float[Array, "... N D"], c: Float[Array, "... M D"]):
        *_, N, Dx = x.shape
        *_, M, Dc = c.shape
        assert Dx == Dc, "x and c should have the same feature dimension"

        x = self.qkv_proj_x(x)
        c = self.qkv_proj_c(c)
        h = jnp.concat([x, c], axis=-2)
        qkv = rearrange(h, "... N (H D) -> ... N H D", H=self.num_heads)
        q, k, v = jnp.split(qkv, 3, axis=-1)
        h = nnx.dot_product_attention(q, k, v)
        h = rearrange(h, "... N H D -> ... N (H D)")
        x, c = jnp.split(h, [N], axis=-2)
        x = self.out_proj_x(x)
        c = self.out_proj_c(c)
        return x, c


class MMDiTBlock(nnx.Module):
    def __init__(self, dim: int, num_heads: int, expand: int = 4, *, rngs: nnx.Rngs):
        self.modulation1_x = Modulation(dim, rngs=rngs)
        self.modulation1_y = Modulation(dim, rngs=rngs)
        self.modulation2_x = Modulation(dim, rngs=rngs)
        self.modulation2_y = Modulation(dim, rngs=rngs)
        self.attention = CrossAttention(dim, num_heads, rngs=rngs)
        self.mlp_x = FeedForward(dim, expand * dim, dim, rngs=rngs)
        self.mlp_y = FeedForward(dim, expand * dim, dim, rngs=rngs)

    def __call__(
        self,
        x: Float[Array, "... N D"],
        y: Float[Array, "... M D"],
        c: Float[Array, "... D"],
    ):
        # cross attention block
        shift_x, scale_x, gate_x = self.modulation1_x(c)
        shift_y, scale_y, gate_y = self.modulation1_y(c)
        hx = adaptive_norm(x, scale_x, shift_x)
        hy = adaptive_norm(y, scale_y, shift_y)
        hx, hy = self.attention(hx, hy)
        x = x + hx * gate_x
        y = y + hy * gate_y

        # feed forward blocks
        shift_x, scale_x, gate_x = self.modulation2_x(c)
        hx = adaptive_norm(x, scale_x, shift_x)
        hx = self.mlp_x(hx)
        x = x + hx * gate_x

        shift_y, scale_y, gate_y = self.modulation2_y(c)
        hy = adaptive_norm(y, scale_y, shift_y)
        hy = self.mlp_y(hy)
        y = y + hy * gate_y
        return x, y


class MMDiT(nnx.Module):
    def __init__(
        self,
        x_dim: int,
        y_dim: int,
        hidden_dim: int,
        num_heads: int,
        num_blocks: int,
        patch_size: int,
        *,
        rngs: nnx.Rngs,
    ):
        self.x_pos_embed = SinusoidalEmbed(hidden_dim, rngs=rngs)
        self.x_embed = FeedForward(x_dim, hidden_dim, hidden_dim, rngs=rngs)

        self.y_pos_embed = SinusoidalEmbed(hidden_dim, rngs=rngs)
        self.y_embed = FeedForward(3*y_dim*patch_size//2, hidden_dim, hidden_dim, rngs=rngs)

        self.c_pos_embed = SinusoidalEmbed(hidden_dim, rngs=rngs)
        self.c_embed = FeedForward(hidden_dim, hidden_dim, hidden_dim, rngs=rngs)

        self.blocks = [
            MMDiTBlock(hidden_dim, num_heads, rngs=rngs) for _ in range(num_blocks)
        ]

        self.out_modulation = Modulation(hidden_dim, rngs=rngs)
        self.out_unembed = FeedForward(hidden_dim, hidden_dim, x_dim, rngs=rngs)

    def __call__(
        self,
        x: Float[Array, "... N D"],
        y: Float[Array, "... M C"],
        t: Float[Array, "..."],
    ) -> Float[Array, "... N D"]:
        # convert to time-frequency domain
        _, _, y = jsp.signal.stft(y, nperseg=PATCH_SIZE - 1, axis=-2, boundary=None)
        y = rearrange(y, "... F C T -> ... T (C F)")
        y = jnp.concatenate([jnp.log(jnp.abs(y)), jnp.cos(jnp.angle(y)), jnp.sin(jnp.angle(y))], axis=-1)

        # embedding
        x_pos = jnp.linspace(0, 1, x.shape[-2])
        x = self.x_embed(x) + self.x_pos_embed(x_pos)
        y_pos = jnp.linspace(0, 1, y.shape[-2])
        y = self.y_embed(y) + self.y_pos_embed(y_pos)
        c = self.c_embed(self.c_pos_embed(t))

        # cross attention blocks
        for block in self.blocks:
            x, y = block(x, y, c)

        # unembedding
        shift, scale, gate = self.out_modulation(c)
        x = adaptive_norm(x, scale, shift)
        x = self.out_unembed(x)
        return x

In [ ]:
# training loop definition
@nnx.jit
def get_train_batch(rngs: nnx.Rngs) -> tuple[Parameters, Scalar, Observation, Parameters]:
    def phi(t: Scalar, x1: Parameters, x0: Parameters) -> Parameters:
        return x1 * t + x0 * (1 - t)

    def train_sample(rng: Key) -> tuple[Parameters, Scalar, Observation, Parameters]:
        rng_xy, rng_x0, rng_t = jr.split(rng, 3)
        x1, y = sample_joint(rng_xy)
        x0 = jr.normal(rng_x0, x1.shape)
        t = jr.uniform(rng_t, minval=0.0, maxval=1.0)

        xt = x1 * t + x0 * (1 - t)
        dx = jax.jacobian(phi)(t, x1, x0)
        return xt, t, y, dx

    return jax.vmap(train_sample)(jr.split(rngs.train(), BATCH_SIZE))


@nnx.jit
def train_step(
    model: MMDiT,
    optimizer: nnx.Optimizer,
    batch: tuple[Parameters, Scalar, Observation, Parameters],
) -> Scalar:
    def loss_fn(model):
        xt, t, y, dx = batch
        return jnp.mean((model(xt, y, t) - dx) ** 2)

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads)
    return loss


flow = MMDiT(
    x_dim=PARAMETERS,
    y_dim=CHANNELS,
    hidden_dim=HIDDEN_DIM,
    num_heads=NUM_HEADS,
    num_blocks=NUM_BLOCKS,
    patch_size=PATCH_SIZE,
    rngs=rngs,
)
optimizer = nnx.Optimizer(flow, optax.adamw(learning_rate=LEARNING_RATE))

for step in (pbar := tqdm(range(TOTAL_EXAMPLES // BATCH_SIZE))):
    batch = get_train_batch(rngs)
    loss = train_step(flow, optimizer, batch)
    pbar.set_postfix(loss=loss.item())

In [ ]:
# evaluation 
RUNS = 1
SAMPLES = 1024 * 1000
DIFFUSIONSTEPS = 1 #16
MCMCWALKERS = 64
MCMCDISCARD = 1000
MCMCTHIN = 10

@nnx.jit
def sample_from_flow(flow: MMDiT, y: Observation, rngs: nnx.Rngs) -> Parameters:
    t = jnp.zeros((SAMPLES,))
    x = jr.normal(rngs.eval(), (SAMPLES, SOURCES, 6))
    y = jnp.broadcast_to(y, (SAMPLES, *y.shape))
    dt = 1.0 / DIFFUSIONSTEPS
    for _ in tqdm(range(DIFFUSIONSTEPS)):
        k1 = flow(x, y, t)
        k2 = flow(x + k1 * dt / 2, y, t + dt / 2)
        k3 = flow(x + k2 * dt / 2, y, t + dt / 2)
        k4 = flow(x + k3 * dt, y, t + dt)
        x = x + (k1 + 2 * k2 + 2 * k3 + k4) * dt / 6
        t += dt
    x = rearrange(x, "... S P -> ... (S P)")
    return x


def sample_from_mcmc(y: Observation, x_true_flat: Parameters):
    p0 = np.random.randn(MCMCWALKERS, x_true_flat.shape[-1])
    sampler = EnsembleSampler(MCMCWALKERS, x_true_flat.shape[-1], log_posterior, args=(y,))
    sampler.run_mcmc(p0, nsteps=MCMCTHIN * SAMPLES // MCMCWALKERS + MCMCDISCARD, progress=True)
    x = sampler.get_chain(flat=True, discard=MCMCDISCARD, thin=10)
    return x


def mirror(x_flat):
    x = rearrange(x_flat, "... (S P) -> ... S P", P=3)
    x_mirrored = x[..., ::-1, :]
    x_mirrored_flat = rearrange(x_mirrored, "... S P -> ... (S P)")
    return x_mirrored_flat


for run in range(RUNS):
    x_true, y = sample_joint(rngs.eval())
    x_true_flat = rearrange(x_true, "... N P -> ... (N P)")

    print("Running flow sampling...")
    generated_samples = np.array(sample_from_flow(flow, y, rngs))
    print("Running MCMC...")
    mcmc_samples = sample_from_mcmc(y, x_true_flat)
    print()

    param_names = sum(([f"$A_{i}$", f"$cos(\\iota_{i})$", f"$f0_{i}$", f"$\\phi0_{i}$", f"$\\theta_{i}$", f"$\\phi_{i}$"] for i in range(SOURCES)), [])
    
    fig = corner(generated_samples, labels=param_names, truths=x_true_flat, color="red")
    fig = corner(mirror(generated_samples), color="orange", fig=fig)
    fig = corner(mcmc_samples, color="blue", fig=fig)
    plt.show()